# Explaining the Misclassification of Class 4 (Septoria Leaf Spot)

This standalone notebook provides six complementary lines of evidence for **why class 4
(Septoria Leaf Spot)** is the hardest class for SimpleNetAugDR-3, and why it is confused
mainly with **class 0 (Bacterial Spot)**, **class 3 (Leaf Mold)**, and **class 2 (Late Blight)**:

1. **Class-4 confusion profile** — which classes the errors fall into.
2. **Grad-CAM** — where the model looks when predicting class 4 (correct vs. wrong cases).
3. **Cross-class Grad-CAM** — on the *same* misclassified leaf, the evidence for the true
   class and for the predicted class overlaps, showing the symptom regions are ambiguous.
4. **Occlusion sensitivity & saliency** — pixel/region-level importance maps.
5. **Feature-space analysis (t-SNE + class-similarity heatmap)** — class 4 overlaps with
   its confusers in the learned representation.
6. **Color-level analysis** — class 4 shares the same yellow-region statistics as classes
   0 and 3, giving a quantitative basis for the paper's "shared yellow features" claim.

**Data note.** This analysis runs on the **held-out test set** (`valid/`). When no pre-trained model is supplied, a model is trained on `train/` with a stratified validation split for early stopping, so the test set is never used during training or model selection — consistent with the benchmark and ablation notebooks.

## 1. Imports & setup

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout, Dense
from tensorflow.keras.utils import to_categorical, normalize
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

RANDOM_SEED = 123
IMG_SIZE    = (128, 128)
NUM_CLASSES = 10
CLASS4      = 4                       # Septoria Leaf Spot

PV_CLASS_NAMES = ['Bacterial Spot', 'Early Blight', 'Late Blight', 'Leaf Mold',
                  'Septoria Leaf Spot', 'Spider Mites', 'Target Spot',
                  'Yellow Leaf Curl Virus', 'Mosaic Virus', 'Healthy']

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print('TensorFlow', tf.__version__, '| ready.')

## 2. Proposed model : SimpleNetAugDR-3

In [ ]:
def create_model():
    model = Sequential([
        Conv2D(64, (4, 4), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (4, 4), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu'),   # SimpleNetAugDR-3 = 64/64/64 (matches Figure 2 & benchmark NB01)
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax'),
    ])
    model.compile(loss='categorical_crossentropy',
                  optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
    return model

print(f'Parameters: {create_model().count_params():,}  (SimpleNetAugDR-3, 64/64/64 = flatten 13x13x64, matches Figure 2)')

## 3. Load PlantVillage (tomato, 128×128) — held-out test set

`valid/` is loaded as the **held-out test set**. Every class-4 analysis below (confusion profile, Grad-CAM, occlusion/saliency, t-SNE, colour) is computed on this set, which the model never sees during training or model selection.

In [ ]:
tomato_train = './new-plant-diseases-dataset/train/'
tomato_valid = './new-plant-diseases-dataset/valid/'


def load_pv(dir_path, img_size=IMG_SIZE):
    X, Y, i = [], [], 0
    for path in tqdm(sorted(os.listdir(dir_path)), desc=os.path.basename(dir_path.rstrip('/'))):
        if not path.startswith('.') and 'Tomato__' in path:
            sub = os.path.join(dir_path, path)
            for f in os.listdir(sub):
                fp = os.path.join(sub, f)
                if not f.startswith('.') and os.path.isfile(fp):
                    img = cv2.imread(fp)
                    img = Image.fromarray(img, 'RGB').resize(img_size)
                    X.append(np.array(img)); Y.append(i)
            i += 1
    return np.array(X, dtype=np.uint8), np.array(Y)


# valid/ = held-out TEST set. The class-4 misclassification analysis runs on this set.
X_test_raw, y_test = load_pv(tomato_valid)
X_test_raw, y_test = shuffle(X_test_raw, y_test, random_state=RANDOM_SEED)
X_test = normalize(X_test_raw.astype(np.float32), axis=1).astype(np.float32)
y_test_cat = to_categorical(y_test, NUM_CLASSES)
print(f'\nHeld-out test set: raw {X_test_raw.shape}, normalised {X_test.shape}')


## 4. Load or train the model

If `MODEL_PATH` points to a saved model (ideally the one trained in the benchmark notebook), it is loaded directly. Otherwise a model is trained here on `train/`, using a stratified **validation** split for early stopping — the held-out **test** set (`valid/`) is reserved for the analysis only.

In [ ]:
MODEL_PATH = ''        # e.g. '/kaggle/input/your-model/best_model.keras' (from the benchmark notebook)

if MODEL_PATH and os.path.exists(MODEL_PATH):
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f'Loaded model from {MODEL_PATH}')
else:
    print('No saved model found -> training on PlantVillage (paper protocol).')
    # Train on train/, holding out a stratified validation split for early stopping.
    X_pool_raw, y_pool = load_pv(tomato_train)
    X_pool = normalize(X_pool_raw.astype(np.float32), axis=1).astype(np.float32)
    del X_pool_raw; gc.collect()
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_pool, y_pool, test_size=0.2, stratify=y_pool, random_state=RANDOM_SEED)
    del X_pool, y_pool; gc.collect()

    model = create_model()
    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(X_tr, to_categorical(y_tr, NUM_CLASSES), batch_size=16, epochs=100,
              validation_data=(X_va, to_categorical(y_va, NUM_CLASSES)),
              callbacks=[es], verbose=1)
    del X_tr, X_va, y_tr, y_va; gc.collect()

# Accuracy on the held-out TEST set (valid/), which the analysis below uses.
test_acc = (model.predict(X_test, verbose=0).argmax(1) == y_test).mean()
print(f'\nHeld-out test accuracy: {test_acc*100:.2f}%')

# Locate the layers Grad-CAM / feature extraction need.
LAST_CONV   = [i for i, l in enumerate(model.layers) if isinstance(l, Conv2D)][-1]
FLATTEN_IDX = [i for i, l in enumerate(model.layers) if isinstance(l, Flatten)][0]
print(f'Last conv layer index: {LAST_CONV} ({model.layers[LAST_CONV].name})')


## 5. Class-4 confusion profile (held-out test set)

In [ ]:
y_prob = model.predict(X_test, verbose=0)
y_pred = y_prob.argmax(1)

cm_norm = confusion_matrix(y_test, y_pred, normalize='true')

# Class-4 row = how true Septoria images are distributed over predicted classes.
row4 = cm_norm[CLASS4]
order = np.argsort(row4)[::-1]
confusers = [c for c in order if c != CLASS4][:3]
print('Class 4 (Septoria) prediction breakdown:')
for c in order:
    tag = '  <-- correct' if c == CLASS4 else ''
    print(f'  -> {PV_CLASS_NAMES[c]:24s} {row4[c]*100:5.1f}%{tag}')
print(f'\nTop confusers: {[PV_CLASS_NAMES[c] for c in confusers]}')

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
im = ax[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax[0].set_xticks(range(10)); ax[0].set_yticks(range(10))
ax[0].set_xticklabels([f'C{i}' for i in range(10)]); ax[0].set_yticklabels([f'C{i}' for i in range(10)])
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True'); ax[0].set_title('Normalised confusion matrix')
ax[0].add_patch(plt.Rectangle((-.5, CLASS4 - .5), 10, 1, fill=False, edgecolor='red', lw=2))
for i in range(10):
    for j in range(10):
        if cm_norm[i, j] > 0.01:
            ax[0].text(j, i, f'{cm_norm[i,j]*100:.0f}', ha='center', va='center',
                       fontsize=7, color='white' if cm_norm[i, j] > 0.5 else 'black')
plt.colorbar(im, ax=ax[0], fraction=0.046)

colors = ['#C44E52' if c == CLASS4 else ('#DD8452' if c in confusers else '#B0B0B0') for c in range(10)]
ax[1].bar([f'C{c}' for c in range(10)], row4 * 100, color=colors)
ax[1].set_title('True Class 4 (Septoria) — predicted-label distribution')
ax[1].set_ylabel('% of class-4 images'); ax[1].set_xlabel('Predicted class')
for c in range(10):
    if row4[c] > 0.01:
        ax[1].text(c, row4[c] * 100, f'{row4[c]*100:.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('class4_confusion_profile.png', dpi=300, bbox_inches='tight')
plt.savefig('class4_confusion_profile.pdf', bbox_inches='tight')
plt.show()
print('Saved -> class4_confusion_profile.png/.pdf')

In [ ]:
#

## 6. Grad-CAM : where the model looks for class 4

Version-robust Grad-CAM via a manual forward pass (works in Keras 2 and 3).

In [ ]:
def grad_cam(model, img, class_idx, last_conv=LAST_CONV):
    """Grad-CAM heatmap for one image (1,H,W,3) and a target class."""
    img = tf.convert_to_tensor(img)
    with tf.GradientTape() as tape:
        x, conv_out = img, None
        for i, layer in enumerate(model.layers):
            x = layer(x, training=False)
            if i == last_conv:
                conv_out = x
                tape.watch(conv_out)
        score = x[:, class_idx]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy()


def overlay(raw_uint8, cam, alpha=0.45):
    cam_r = cv2.resize(cam, (raw_uint8.shape[1], raw_uint8.shape[0]))
    heat = (cm.jet(cam_r)[..., :3] * 255).astype(np.uint8)
    return (alpha * heat + (1 - alpha) * raw_uint8).astype(np.uint8)


# Class-4 images, split into correctly classified and misclassified.
idx4 = np.where(y_test == CLASS4)[0]
correct4 = [i for i in idx4 if y_pred[i] == CLASS4]
wrong4   = [i for i in idx4 if y_pred[i] != CLASS4]
print(f'Class-4 images: {len(idx4)} | correct: {len(correct4)} | misclassified: {len(wrong4)}')

sel_c = correct4[:4]
sel_w = wrong4[:4]
rng = np.random.default_rng(RANDOM_SEED)
if len(sel_c) < 4: sel_c = list(rng.choice(idx4, 4, replace=False))

fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for col, i in enumerate(sel_c):
    cam = grad_cam(model, X_test[i:i+1], CLASS4)
    axes[0, 2*col].imshow(X_test_raw[i]);            axes[0, 2*col].set_title(f'OK  p={y_prob[i, CLASS4]:.2f}', fontsize=9)
    axes[0, 2*col+1].imshow(overlay(X_test_raw[i], cam)); axes[0, 2*col+1].set_title('Grad-CAM (C4)', fontsize=9)
for col, i in enumerate(sel_w):
    cam = grad_cam(model, X_test[i:i+1], CLASS4)
    axes[1, 2*col].imshow(X_test_raw[i]);            axes[1, 2*col].set_title(f'WRONG->C{y_pred[i]}', fontsize=9, color='red')
    axes[1, 2*col+1].imshow(overlay(X_test_raw[i], cam)); axes[1, 2*col+1].set_title('Grad-CAM (C4)', fontsize=9)
for ax in axes.ravel(): ax.axis('off')
axes[0, 0].set_ylabel('correct', fontsize=11)
fig.suptitle('Grad-CAM for Class 4 (Septoria Leaf Spot): correctly classified (top) vs misclassified (bottom)', y=1.02)
plt.tight_layout()
plt.savefig('class4_gradcam.png', dpi=300, bbox_inches='tight')
plt.savefig('class4_gradcam.pdf', bbox_inches='tight')
plt.show()
print('Saved -> class4_gradcam.png/.pdf')

## 7. Cross-class Grad-CAM on misclassified leaves

For a leaf that is truly class 4 but predicted as a confuser, we show the Grad-CAM for the **true** class 4 and for the **predicted** class. Overlapping attention indicates the symptom regions are genuinely ambiguous.

In [ ]:
examples = wrong4[:4] if len(wrong4) >= 4 else list(rng.choice(idx4, 4, replace=False))

fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
if len(examples) == 1: axes = axes[None, :]
for r, i in enumerate(examples):
    p = int(y_pred[i])
    cam_true = grad_cam(model, X_test[i:i+1], CLASS4)
    cam_pred = grad_cam(model, X_test[i:i+1], p)
    axes[r, 0].imshow(X_test_raw[i]); axes[r, 0].set_title(f'leaf (true C4, pred C{p})', fontsize=9)
    axes[r, 1].imshow(overlay(X_test_raw[i], cam_true)); axes[r, 1].set_title(f'evidence for C4 (p={y_prob[i,CLASS4]:.2f})', fontsize=9)
    axes[r, 2].imshow(overlay(X_test_raw[i], cam_pred)); axes[r, 2].set_title(f'evidence for C{p} (p={y_prob[i,p]:.2f})', fontsize=9)
    for c in range(3): axes[r, c].axis('off')
fig.suptitle('Same leaf, two explanations: class-4 and predicted-class evidence overlap on the lesion regions', y=1.005)
plt.tight_layout()
plt.savefig('class4_crossclass_gradcam.png', dpi=300, bbox_inches='tight')
plt.savefig('class4_crossclass_gradcam.pdf', bbox_inches='tight')
plt.show()
print('Saved -> class4_crossclass_gradcam.png/.pdf')

## 8. Occlusion sensitivity & saliency maps

Two gradient-free / gradient-based region-importance views that corroborate Grad-CAM.

In [ ]:
def occlusion_map(model, img_norm, class_idx, patch=16, stride=8):
    H, W = img_norm.shape[:2]
    base = float(img_norm.mean())
    variants, coords = [], []
    for y in range(0, H - patch + 1, stride):
        for x in range(0, W - patch + 1, stride):
            v = img_norm.copy(); v[y:y+patch, x:x+patch, :] = base
            variants.append(v); coords.append((y, x))
    variants = np.array(variants, dtype=np.float32)
    probs = model.predict(variants, verbose=0)[:, class_idx]
    heat = np.zeros((H, W), np.float32); cnt = np.zeros((H, W), np.float32)
    for (y, x), pr in zip(coords, probs):
        heat[y:y+patch, x:x+patch] += (1 - pr); cnt[y:y+patch, x:x+patch] += 1
    heat /= np.maximum(cnt, 1e-8)
    return (heat - heat.min()) / (np.ptp(heat) + 1e-8)


def saliency_map(model, img_norm, class_idx):
    x = tf.convert_to_tensor(img_norm[None].astype(np.float32))
    with tf.GradientTape() as tape:
        tape.watch(x)
        score = model(x, training=False)[:, class_idx]
    g = tf.abs(tape.gradient(score, x))[0].numpy().max(-1)
    return (g - g.min()) / (np.ptp(g) + 1e-8)


demo = (wrong4[:3] if len(wrong4) >= 3 else list(rng.choice(idx4, 3, replace=False)))
fig, axes = plt.subplots(len(demo), 4, figsize=(12, 3 * len(demo)))
if len(demo) == 1: axes = axes[None, :]
for r, i in enumerate(demo):
    occ = occlusion_map(model, X_test[i], CLASS4)
    sal = saliency_map(model, X_test[i], CLASS4)
    cam = grad_cam(model, X_test[i:i+1], CLASS4)
    axes[r, 0].imshow(X_test_raw[i]);                axes[r, 0].set_title(f'true C4 -> pred C{y_pred[i]}', fontsize=9)
    axes[r, 1].imshow(overlay(X_test_raw[i], cam));  axes[r, 1].set_title('Grad-CAM', fontsize=9)
    axes[r, 2].imshow(overlay(X_test_raw[i], occ));  axes[r, 2].set_title('Occlusion', fontsize=9)
    axes[r, 3].imshow(sal, cmap='hot');             axes[r, 3].set_title('Saliency', fontsize=9)
    for c in range(4): axes[r, c].axis('off')
fig.suptitle('Region-importance for Class 4 — Grad-CAM, occlusion and saliency agree on the lesion areas', y=1.005)
plt.tight_layout()
plt.savefig('class4_occlusion_saliency.png', dpi=300, bbox_inches='tight')
plt.savefig('class4_occlusion_saliency.pdf', bbox_inches='tight')
plt.show()
print('Saved -> class4_occlusion_saliency.png/.pdf')

## 9. Feature-space analysis (t-SNE) & class-similarity heatmap

If class 4 overlaps its confusers in the learned representation, the classes are intrinsically hard to separate.

In [ ]:
def extract_features(model, X, upto=FLATTEN_IDX, batch=256):
    feats = []
    for i in range(0, len(X), batch):
        x = tf.convert_to_tensor(X[i:i+batch])
        for j, layer in enumerate(model.layers):
            x = layer(x, training=False)
            if j == upto: break
        feats.append(x.numpy())
    return np.concatenate(feats)


# Balanced sample for speed.
per = 150
samp = np.concatenate([np.where(y_test == c)[0][:per] for c in range(10)])
F = extract_features(model, X_test[samp])
ys = y_test[samp]
print(f'Feature matrix: {F.shape}')

Fp = PCA(n_components=50, random_state=RANDOM_SEED).fit_transform(F)
Z = TSNE(n_components=2, perplexity=30, init='pca', random_state=RANDOM_SEED).fit_transform(Fp)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
# t-SNE: grey background, highlight class 4 and its top confusers.
hi = [CLASS4] + list(confusers)
palette = {CLASS4: '#C44E52', confusers[0]: '#DD8452', confusers[1]: '#4C72B0', confusers[2]: '#55A868'}
ax[0].scatter(Z[~np.isin(ys, hi), 0], Z[~np.isin(ys, hi), 1], s=10, c='#D5D5D5', label='other')
for c in hi:
    m = ys == c
    ax[0].scatter(Z[m, 0], Z[m, 1], s=18, c=palette[c],
                  label=f'C{c} {PV_CLASS_NAMES[c]}', edgecolors='k', linewidths=0.2)
ax[0].legend(fontsize=8, loc='best'); ax[0].set_title('t-SNE of learned features (class 4 + confusers)')
ax[0].set_xticks([]); ax[0].set_yticks([])

# Class-similarity heatmap from feature centroids.
cent = np.array([F[ys == c].mean(0) for c in range(10)])
S = cosine_similarity(cent)
im = ax[1].imshow(S, cmap='magma')
ax[1].set_xticks(range(10)); ax[1].set_yticks(range(10))
ax[1].set_xticklabels([f'C{i}' for i in range(10)]); ax[1].set_yticklabels([f'C{i}' for i in range(10)])
ax[1].set_title('Cosine similarity of class feature centroids')
ax[1].add_patch(plt.Rectangle((-.5, CLASS4 - .5), 10, 1, fill=False, edgecolor='cyan', lw=2))
for i in range(10):
    for j in range(10):
        ax[1].text(j, i, f'{S[i,j]:.2f}', ha='center', va='center', fontsize=6,
                   color='white' if S[i, j] < 0.6 else 'black')
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout()
plt.savefig('class4_feature_space.png', dpi=300, bbox_inches='tight')
plt.savefig('class4_feature_space.pdf', bbox_inches='tight')
plt.show()

sim4 = S[CLASS4].copy(); sim4[CLASS4] = -1
nearest = np.argsort(sim4)[::-1][:3]
print('Most similar classes to C4 in feature space:',
      [f'{PV_CLASS_NAMES[c]} ({S[CLASS4,c]:.2f})' for c in nearest])
print('Saved -> class4_feature_space.png/.pdf')

## 10. Color-level analysis : the shared yellow signature

The paper attributes the confusion to shared yellow regions. We quantify it: the fraction of yellow pixels per class.

In [ ]:
def yellow_fraction(img_uint8):
    hsv = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    mask = (h >= 20) & (h <= 38) & (s > 40) & (v > 40)   # OpenCV hue scale 0-179
    return float(mask.mean())


per = 200
yf = {}
for c in range(10):
    ids = np.where(y_test == c)[0][:per]
    yf[c] = np.mean([yellow_fraction(X_test_raw[i]) for i in ids])

order_y = sorted(range(10), key=lambda c: yf[c], reverse=True)
print('Yellow-pixel fraction per class (high -> low):')
for c in order_y:
    print(f'  C{c} {PV_CLASS_NAMES[c]:24s} {yf[c]*100:5.1f}%')

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
cols = ['#C44E52' if c == CLASS4 else ('#DD8452' if c in confusers else '#B0B0B0') for c in range(10)]
ax[0].bar([f'C{c}' for c in range(10)], [yf[c] * 100 for c in range(10)], color=cols)
ax[0].set_title('Yellow-pixel fraction per class (C4 red, confusers orange)')
ax[0].set_ylabel('% yellow pixels'); ax[0].set_xlabel('Class')
for c in range(10):
    ax[0].text(c, yf[c] * 100, f'{yf[c]*100:.0f}', ha='center', va='bottom', fontsize=8)

# Example montage: C4 vs its confusers.
montage = [CLASS4] + list(confusers[:2])
ax[1].axis('off')
sub = fig.add_subplot(1, 2, 2)
sub.axis('off'); sub.set_title('Example leaves: C4 vs confusers (visual similarity)')
grid = np.ones((2 * 128 + 8, len(montage) * 128 + (len(montage) - 1) * 4, 3), np.uint8) * 255
for col, c in enumerate(montage):
    ids = np.where(y_test == c)[0][:2]
    for row, i in enumerate(ids):
        y0 = row * (128 + 8); x0 = col * (128 + 4)
        grid[y0:y0 + 128, x0:x0 + 128] = X_test_raw[i]
sub.imshow(grid)
for col, c in enumerate(montage):
    sub.text(col * (128 + 4) + 64, -6, f'C{c}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('class4_color_analysis.png', dpi=300, bbox_inches='tight')
plt.savefig('class4_color_analysis.pdf', bbox_inches='tight')
plt.show()
print('Saved -> class4_color_analysis.png/.pdf')